# Stochastic Unit Commitment example: PyPSA → SMS++

This notebook builds a small PyPSA network directly in Python, converts it into a stochastic network, solves the two-stage stochastic problem with PyPSA, and then solves the same instance through the `pypsa2smspp` transformation pipeline.

The example  assumes that `pypsa2smspp`, `pySMSpp`, SMS++, PyPSA, and the selected solver are already installed and correctly configured.


In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import pypsa

from pypsa2smspp.transformation import Transformation
from pypsa2smspp.network_correction import add_slack_unit

## Settings

The example uses three equiprobable scenarios. Both demand and renewable availability are scenario-dependent.

In [12]:
CASE_NAME = "stochastic_uc_example"

SOLVER_NAME = "gurobi"
SOLVER_OPTIONS = {
    "Threads": 32,
    "Method": 2,
    "Crossover": 0,
    "Seed": 123,
    "AggFill": 0,
    "PreDual": 0,
}

SCENARIO_PROBABILITIES = {
    "low": 1 / 3,
    "med": 1 / 3,
    "high": 1 / 3,
}

STOCHASTIC_PARAMETERS = ["demand", "renewables"]

## Build the deterministic base network

The network has two AC buses, one transmission line, two loads, two renewable generators, and one committable gas generator.

The gas generator makes this a unit-commitment example. Demand profiles are generated directly in the notebook.

In [ ]:
snapshots = pd.date_range("2025-01-01 00:00", periods=24, freq="h")

n = pypsa.Network()
n.set_snapshots(snapshots)

# Keep snapshot weights explicit.
n.snapshot_weightings.loc[:, :] = 1.0

# Carriers
n.add("Carrier", "AC")
n.add("Carrier", "solar")
n.add("Carrier", "onwind")
n.add("Carrier", "slack")

# Buses
n.add("Bus", "IT0 0", carrier="AC", v_nom=380)
n.add("Bus", "IT0 1", carrier="AC", v_nom=380)

# Transmission line
n.add(
    "Line",
    "Line 0-1",
    bus0="IT0 0",
    bus1="IT0 1",
    carrier="AC",
    x=0.1,
    r=0.01,
    s_nom=700,
)

# Loads
n.add("Load", "load_IT0_0", bus="IT0 0", carrier="AC")
n.add("Load", "load_IT0_1", bus="IT0 1", carrier="AC")

# Renewable generators
n.add(
    "Generator",
    "solar_IT0_0",
    bus="IT0 0",
    carrier="solar",
    p_nom=45,
    p_min_pu=0,
    p_max_pu=1,
    marginal_cost=0.01,
    p_nom_extendable=True,
    capital_cost=55
)

n.add(
    "Generator",
    "wind_IT0_1",
    bus="IT0 1",
    carrier="onwind",
    p_nom=35,
    p_min_pu=0,
    p_max_pu=1,
    marginal_cost=0.015,
    p_nom_extendable=True,
    capital_cost=70
)

# Deterministic base demand.
hours = np.arange(len(snapshots))

load_0 = 220 + 60 * np.sin(2 * np.pi * (hours - 7) / 24)
load_1 = 260 + 70 * np.sin(2 * np.pi * (hours - 8) / 24)
load_0 = np.maximum(load_0, 150)
load_1 = np.maximum(load_1, 180)

n.loads_t.p_set = pd.DataFrame(
    {
        "load_IT0_0": load_0,
        "load_IT0_1": load_1,
    },
    index=snapshots,
)

# Deterministic renewable availability.
solar_profile = np.maximum(0, np.sin(np.pi * (hours - 6) / 12))
wind_profile = 0.45 + 0.20 * np.sin(2 * np.pi * (hours + 3) / 24)
wind_profile = np.clip(wind_profile, 0.10, 0.85)

n.generators_t.p_max_pu = pd.DataFrame(
    {
        "solar_IT0_0": solar_profile,
        "wind_IT0_1": wind_profile,
    },
    index=snapshots,
)

# Add high-cost slack units to guarantee feasibility in all scenarios.
n = add_slack_unit(n)

n

PyPSA Network 'Unnamed Network'
-------------------------------
Components:
 - Bus: 2
 - Carrier: 4
 - Generator: 4
 - Line: 1
 - Load: 2
Snapshots: 24

## Convert the network to a stochastic PyPSA network

The deterministic profiles are copied before calling `set_scenarios`. Then each scenario receives its own demand and renewable availability.

In [14]:
base_load = n.loads_t.p_set.copy()
base_p_max_pu = n.generators_t.p_max_pu.copy()

n.set_scenarios(SCENARIO_PROBABILITIES)

# Scenario-dependent demand.
demand_multiplier = {
    "low": 0.85,
    "med": 1.00,
    "high": 1.20,
}

# Scenario-dependent renewable availability.
renewable_multiplier = {
    "low": 0.70,
    "med": 0.85,
    "high": 1.00,
}

for scenario in SCENARIO_PROBABILITIES:
    n.loads_t.p_set[scenario] = base_load * demand_multiplier[scenario]
    n.generators_t.p_max_pu[scenario] = (base_p_max_pu * renewable_multiplier[scenario]).clip(upper=1.0)

n

Stochastic PyPSA Network 'Unnamed Network'
------------------------------------------
Components:
 - Bus: 6
 - Carrier: 12
 - Generator: 12
 - Line: 3
 - Load: 6
Snapshots: 24
Scenarios: 3

## Solve the stochastic unit-commitment problem with PyPSA

This is the PyPSA reference solution. The solve uses normal unit commitment, not linearized unit commitment.

In [ ]:
n_pypsa = n.copy()

n_pypsa.optimize(
    solver_name=SOLVER_NAME,
    solver_options=SOLVER_OPTIONS,
)

obj_pypsa = n_pypsa.objective + n_pypsa.objective_constant

statistics_pypsa = n_pypsa.statistics()

print(f"PyPSA objective: {obj_pypsa}")
statistics_pypsa

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - Threads: 32
 - Method: 2
 - Crossover: 0
 - Seed: 123
 - AggFill: 0
 - PreDual: 0
INFO:linopy.io: Writing time: 0.19s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2771537


INFO:gurobipy:Set parameter LicenseID to value 2771537


...

INFO:gurobipy:Barrier solved model in 0 iterations and 0.01 seconds (0.00 work units)


Optimal objective 5.59363178e+07


INFO:gurobipy:Optimal objective 5.59363178e+07
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 360 primals, 864 duals
Objective: 5.59e+07
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


PyPSA objective: 55936317.76758582


Optimal Capacity  Installed Capacity      Supply  \
Generator high onwind         350.00000           350.00000  3780.00000   
               slack        12159.11099         12159.11099  6625.91065   
               solar          450.00000           450.00000  3418.08935   
          low  onwind         350.00000           350.00000  2646.00000   
               slack        12159.11099         12159.11099  4753.33745   
               solar          450.00000           450.00000  2392.66255   
          med  onwind         350.00000           350.00000  3213.00000   
               slack        12159.11099         12159.11099  5401.62405   
               solar          450.00000           450.00000  2905.37595   
Line      high AC             700.00000           700.00000  2499.40646   
          low  AC             700.00000           700.00000   676.38070   
          med  AC             700.00000           700.00000   949.44264   

                       Withdrawal  Energy Balance  Transmission  \
Generator high onwind     0.00000      3780.00000       0.00000   
               slack      0.00000      6625.91065       0.00000   
               solar      0.00000      3418.08935       0.00000   
          low  onwind     0.00000      2646.00000       0.00000   
               slack      0.00000      4753.33745       0.00000   
               solar      0.00000      2392.66255       0.00000   
          med  onwind     0.00000      3213.00000       0.00000   
               slack      0.00000      5401.62405       0.00000   
               solar      0.00000      2905.37595       0.00000   
Line      high AC      1148.28741      1351.11905   -1351.11905   
          low  AC      1657.07521      -980.69451     980.69451   
          med  AC      1933.71512      -984.27248     984.27248   

                       Capacity Factor   Curtailment  Capital Expenditure  \
Generator high onwind         0.450000       0.00000                  0.0   
               slack          0.022706  285192.75315                  0.0   
               solar          0.316490       0.00000                  0.0   
          low  onwind         0.315000       0.00000                  0.0   
               slack          0.016289  287065.32634                  0.0   
               solar          0.221543       0.00000                  0.0   
          med  onwind         0.382500       0.00000                  0.0   
               slack          0.018510  286417.03975                  0.0   
               solar          0.269016       0.00000                  0.0   
Line      high AC             0.217125       0.00000                  0.0   
          low  AC             0.138896       0.00000                  0.0   
          med  AC             0.171617       0.00000                  0.0   

                       Operational Expenditure       Revenue  Market Value  
Generator high onwind             5.670000e+01  1.260000e+07   3333.333333  
               slack              6.625911e+07  2.208637e+07   3333.333338  
               solar              3.418089e+01  1.139363e+07   3333.333324  
          low  onwind             3.969000e+01  8.820000e+06   3333.333333  
               slack              4.753337e+07  1.584446e+07   3333.333287  
               solar              2.392663e+01  7.975542e+06   3333.333425  
          med  onwind             4.819500e+01  1.071000e+07   3333.333333  
               slack              5.401624e+07  1.800541e+07   3333.333316  
               solar              2.905376e+01  9.684586e+06   3333.333366  
Line      high AC                 0.000000e+00  4.503730e+06   1801.919795  
          low  AC                 0.000000e+00 -3.268982e+06  -4833.049778  
          med  AC                 0.000000e+00 -3.280908e+06  -3455.615031

## Build and solve the SMS++ model

The transformation is configured for a Two-Stage Stochastic Block (`tssb`) with stochastic demand and renewable availability.

In [ ]:
n_smspp = n.copy()

transformation = Transformation(
    name=CASE_NAME,
    configfile="TSSBlock/TSSBSCfg_grb.txt",
    capacity_expansion_ucblock=True,
    stochastic_parameters={
        "stochastic_type": "tssb",
        "parameters": STOCHASTIC_PARAMETERS,
    },
)

smspp_network = transformation.create_model(n_smspp, verbose=True)

smspp_network.print_tree()

[START] consistency_check
[ OK  ] consistency_check (0.000s)
[START] direct


[2026-05-08 07:11:42,937] INFO - pypsa2smspp - No links found: skipping dynamic-to-static link preprocessing.


[ OK  ] direct (0.050s)
[START] prepare_tssb_interface
[ OK  ] prepare_tssb_interface (0.045s)
[START] convert_to_blocks
[ OK  ] convert_to_blocks (0.134s)
SMSNetwork
└── Block_0 [TwoStageStochasticBlock]
    ├── DiscreteScenarioSet [DiscreteScenarioSet]
    ├── StaticAbstractPath [AbstractPath]
    └── StochasticBlock [StochasticBlock]
        ├── AbstractPath [AbstractPath]
        └── Block [UCBlock]
            ├── UnitBlock_0 [IntermittentUnitBlock]
            ├── UnitBlock_1 [IntermittentUnitBlock]
            ├── UnitBlock_2 [SlackUnitBlock]
            └── UnitBlock_3 [SlackUnitBlock]


In [17]:
transformation.optimize(verbose=True)

print(f"SMS++ objective: {transformation.result.objective_value}")

[START] optimize
Executing command:
tssb_solver temp_stochastic_uc_example.nc -S TSSBSCfg_grb.txt -c /home/pampado/pySMSpp/pysmspp/data/configs/TSSBlock/ -p /home/pampado/pypsa2smspp/docs/examples/output/ -O /home/pampado/pypsa2smspp/docs/examples/output/solution_stochastic_uc_example.nc

temp_stochastic_uc_example.nc is a Block file
DEBUG [apply_baseline_selection]: Selected 3 scenarios using baseline method (top weights)
Set parameter LogToConsole to value 1

Peak CPU Usage: 49.80 %
Peak Memory Usage: 66.07 MB
Total Time: 1.31 seconds

[FAIL ] optimize (1.323s) -> Failed to run tssb_solver with error log:


Full log:
temp_stochastic_uc_example.nc is a Block file
DEBUG [apply_baseline_selection]: Selected 3 scenarios using baseline method (top weights)
Set parameter LogToConsole to value 1
Peak CPU Usage: 49.80 %
Peak Memory Usage: 66.07 MB
Total Time: 1.31 seconds



ValueError: Failed to run tssb_solver with error log:


Full log:
temp_stochastic_uc_example.nc is a Block file
DEBUG [apply_baseline_selection]: Selected 3 scenarios using baseline method (top weights)
Set parameter LogToConsole to value 1
Peak CPU Usage: 49.80 %
Peak Memory Usage: 66.07 MB
Total Time: 1.31 seconds


## Retrieve the SMS++ solution back into PyPSA

In [ ]:
n_smspp = transformation.retrieve_solution(n_smspp, verbose=True)

obj_smspp = transformation.result.objective_value
statistics_smspp = n_smspp.statistics()

relative_error_pct = (obj_smspp - obj_pypsa) / obj_pypsa * 100

print(f"PyPSA objective: {obj_pypsa}")
print(f"SMS++ objective: {obj_smspp}")
print(f"Relative error SMS++ vs PyPSA: {relative_error_pct:.6f}%")

statistics_smspp

## Print the summary

In [ ]:

summary = pd.DataFrame(
    [
        {
            "case": CASE_NAME,
            "objective_pypsa": obj_pypsa,
            "objective_smspp": obj_smspp,
            "relative_error_pct": relative_error_pct,
            "n_snapshots": len(n.snapshots),
            "n_scenarios": len(SCENARIO_PROBABILITIES),
            "stochastic_parameters": ", ".join(STOCHASTIC_PARAMETERS),
        }
    ]
)

summary